In [28]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
)
import plotly.express as px


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import (
    ALL_CHAINS,
    ROOT_PRICE_ORACLE,
    ChainData,
    STATS_CALCULATOR_REGISTRY,
    WETH,
    ETH_CHAIN,
)

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks,
)


import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationStates,
    DestinationTokenValues,
    AutopoolDestinationStates,
    Autopools,
    DestinationTokens,
    Destinations,
    Tokens,
)
from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_df,
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_highest_value_in_field_where,
    get_subset_not_already_in_column,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    get_state_by_one_block,
)
from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_active_destinations_by_autopool_by_block,
    fetch_pools_and_destinations_df,
)
from mainnet_launch.constants import (
    AutopoolConstants,
    ALL_AUTOPOOLS,
    AUTO_LRT,
    POINTS_HOOK,
    ChainData,
)


def build_autopool_balance_of_calls_by_destination(
    autopool_vault_address: str, destination_vault_addresses: list[str]
) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["balanceOf(address)(uint256)", autopool_vault_address],
            [((autopool_vault_address, destination_vault_address, "balanceOf"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


def fetch_autopool_balance_of_by_destination(
    autopool_to_all_ever_active_destinations: dict[str, list[Destinations]], missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopool_balance_of_calls = []

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]

        autopool_balance_of_calls.extend(
            build_autopool_balance_of_calls_by_destination(autopool_vault_address, this_autopool_active_destinations)
        )

    autopool_destination_balance_of_df = get_raw_state_by_blocks(
        autopool_balance_of_calls, missing_blocks, chain, include_block_number=True
    )

    autopool_destination_balance_of_records = []

    def _extract_autopool_destination_vault_balance_of_block(row: dict):
        for k in row.keys():
            if k != "block":
                autopool_vault_address, destination_vault_address, _ = k
                balance_of = row[k]
                autopool_destination_balance_of_records.append(
                    {
                        "block": row["block"],
                        "autopool_vault_address": autopool_vault_address,
                        "destination_vault_address": destination_vault_address,
                        "balance_of": balance_of,
                    }
                )

    autopool_destination_balance_of_df.apply(_extract_autopool_destination_vault_balance_of_block, axis=1)
    # returns a table of (block, autopool_vault_address, destination_vault_address, balance_of)
    return pd.DataFrame.from_records(autopool_destination_balance_of_records)


def ensure_autopool_destination_states_is_current():
    for chain in ALL_CHAINS:
        possible_blocks = build_blocks_to_use(chain)

        missing_blocks = get_subset_not_already_in_column(
            AutopoolDestinationStates,
            AutopoolDestinationStates.block,
            possible_blocks,
            where_clause=AutopoolDestinationStates.chain_id == chain.chain_id,
        )

        if len(missing_blocks) == 0:
            continue

        token_value_df = natural_left_right_using_where(
            DestinationTokenValues,
            TokenValues,
            using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
            where_clause=DestinationTokenValues.chain_id == chain.chain_id,
        )

        token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)[
            ["symbol", "decimals", "token_address"]
        ]
        token_value_df = pd.merge(token_value_df, token_df, how="left", on="token_address")

        autopool_to_all_ever_active_destinations = (
            fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks(chain, missing_blocks)
        )

        autopool_destination_balance_of_df = fetch_autopool_balance_of_by_destination(
            autopool_to_all_ever_active_destinations, missing_blocks, chain
        )

        destination_states_df = get_full_table_as_df(
            DestinationStates, where_clause=DestinationStates.chain_id == chain.chain_id
        )

        return (
            destination_states_df,
            autopool_destination_balance_of_df,
            autopool_to_all_ever_active_destinations,
            token_value_df,
        )

        # insert_avoid_conflicts(
        #     all_new_destination_states,
        #     DestinationStates,
        #     index_elements=[
        #         DestinationStates.block,
        #         DestinationStates.chain_id,
        #         DestinationStates.destination_vault_address,
        #     ],
        # )


destination_states_df, autopool_destination_balance_of_df, autopool_to_all_ever_active_destinations, token_value_df = (
    ensure_autopool_destination_states_is_current()
)

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]


In [35]:
limited_destination_states_df = destination_states_df[
    [
        "destination_vault_address",
        "block",
        "underlying_token_total_supply",
        "underlying_safe_price",
        "underlying_spot_price",
        "underlying_backing",
        "chain_id",
    ]
].copy()
autopool_destination_balance_of_df

,block,autopool_vault_address,destination_vault_address,balance_of
0,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0.000000
1,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0x865e59D439BF7310c9BC6117E6020B8C87De4065,0.000000
2,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,0.000000
3,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2,0.000000
4,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1,0.000000
...,...,...,...,...
15003,22356658.0,0xE800e3760FC20aA98c5df6A9816147f190455AF3,0x60339056EC88996e41757E05a798310E46972cca,0.000000
15004,22356658.0,0xE800e3760FC20aA98c5df6A9816147f190455AF3,0x5c6aeb9ef0d5BbA4E6691f381003503FD0D45126,470.519208
15005,22356658.0,0x35911af1B570E26f668905595dEd133D01CD3E5a,0xc4Eb861e7b66f593482a3D7E8adc314f6eEDA30B,317.661117
15006,22356658.0,0x35911af1B570E26f668905595dEd133D01CD3E5a,0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04,0.000000


In [39]:
raw_autopool_destination_state_df = pd.merge(
    limited_destination_states_df, autopool_destination_balance_of_df, on=["block", "destination_vault_address"]
)

new_autopool_destination_state_rows = []


def _extract_autopool_destination_state(row: dict) -> None:
    new_autopool_destination_state_rows.append(
        AutopoolDestinationStates(
            destination_vault_address=row["destination_vault_address"],
            autopool_vault_address=row["autopool_vault_address"],
            block=row["block"],
            chain_id=row["chain_id"],
            amount=row["balance_of"],
            total_safe_value=row["balance_of"] * row["underlying_safe_price"],
            total_spot_value=row["balance_of"] * row["underlying_spot_price"],
            total_backing_value=row["balance_of"] * row["underlying_backing"],
            percent_ownership=100 * (row["balance_of"] / row["underlying_token_total_supply"]),
        )
    )


raw_autopool_destination_state_df.apply(_extract_autopool_destination_state, axis=1)

0        None
1        None
2        None
3        None
4        None
         ... 
15003    None
15004    None
15005    None
15006    None
15007    None
Length: 15008, dtype: object

In [ ]:
insert_avoid_conflicts(
    new_autopool_destination_state_rows,
    AutopoolDestinationStates,
    index_elements=[
        AutopoolDestinationStates.destination_vault_address,
        AutopoolDestinationStates.autopool_vault_address,
        AutopoolDestinationStates.block,
        AutopoolDestinationStates.chain_id,
    ],
)

In [25]:
destination_states_df

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,underlying_safe_price,underlying_spot_price,underlying_backing,safe_backing_discount,spot_backing_discount,safe_spot_spread
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,None,0.046281,0.046281,13157.451200,12302.611124,1.036523,-0.000645,1.036523,1.036516,1.035855,0.000645,0.000638,6.621342e-06
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,1,0.040938,0.003618,0.005802,0.0,None,0.045987,0.045987,13157.496413,12383.288934,1.036163,-0.000278,1.036163,1.036121,1.035875,0.000278,0.000237,4.066212e-05
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,1,0.040507,0.003295,0.005848,0.0,None,0.045061,0.045061,13157.496413,12383.288934,1.036462,-0.000538,1.036462,1.036456,1.035906,0.000537,0.000531,6.483657e-06
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,1,0.040196,0.003022,0.005817,0.0,None,0.044440,0.044440,13268.458946,12479.834854,1.036491,-0.000575,1.036491,1.036477,1.035896,0.000574,0.000560,1.429726e-05
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,1,0.040448,0.002794,0.005885,0.0,None,0.044524,0.044524,13268.526874,12494.174709,1.036501,-0.000558,1.036501,1.036502,1.035924,0.000557,0.000558,-6.380341e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8507,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22327989,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,5048.902326,0.000000,1.045577,0.000000,1.045577,1.045772,1.046107,-0.000506,-0.000320,-1.863948e-04
8508,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22335153,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,5048.902326,0.000000,1.045815,0.000000,1.045815,1.045912,1.046169,-0.000338,-0.000246,-9.195140e-05
8509,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22342307,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,5117.872032,0.000000,1.045629,0.000000,1.045629,1.045839,1.046240,-0.000584,-0.000383,-2.012162e-04
8510,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22349480,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,4975.260393,0.000000,1.045822,0.000000,1.045822,1.045994,1.046303,-0.000459,-0.000295,-1.649598e-04


In [7]:
token_value_df[["block", "token_address", "destination_vault_address"]]

,block,token_address,destination_vault_address
0,20759464,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0x4a712BBFD80480eaEd9999DA8971B78DB2E12230
1,20759464,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0x5996333aDf54e3aC8C0Fc3De0fe51de0B75BC00d
2,20759464,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0x65efCF2cce562DCBf07e805eEbeDeF21Dbd8Ea3D
3,20759464,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0x6F628dcCD275feA4277722D177265741031E09d7
4,20759464,0xf939E0A03FB07F59A73314E73794Be0E57ac1b4E,0x7583b1589aDD33320366A48A92794D77763FAE9e
...,...,...,...
35835,22356658,0xdAC17F958D2ee523a2206206994597C13D831ec7,0xd7900d87069C815a299bdA7aFDcd7eEe98fe4b6c
35836,22356658,0x853d955aCEf822Db058eb8505911ED77F175b99e,0x9906eB64BA32Fb4fD3e5541ecA95e57610084d02
35837,22356658,0x853d955aCEf822Db058eb8505911ED77F175b99e,0xEeEE628a00B0EDa99D1bc78a72Fa2F2Df4AdE2F4
35838,22356658,0xA663B02CF0a4b149d2aD41910CB81e23e1c41c32,0x4686CEe2B3b1Da28a1db926443d5DDd674462b29


In [8]:
destination_states_df.columns

Index(['destination_vault_address', 'block', 'chain_id', 'incentive_apr',
       'fee_apr', 'base_apr', 'points_apr', 'fee_plus_base_apr',
       'total_apr_in', 'total_apr_out', 'underlying_token_total_supply',
       'safe_total_supply', 'price_per_share', 'price_return',
       'underlying_safe_price', 'underlying_spot_price', 'underlying_backing',
       'safe_backing_discount', 'spot_backing_discount', 'safe_spot_spread'],
      dtype='object')

In [ ]:
limited_destination_states_df

,destination_vault_address,block,underlying_token_total_supply,underlying_safe_price,underlying_spot_price,underlying_backing
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,13157.451200,1.036523,1.036516,1.035855
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,13157.496413,1.036163,1.036121,1.035875
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,13157.496413,1.036462,1.036456,1.035906
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,13268.458946,1.036491,1.036477,1.035896
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,13268.526874,1.036501,1.036502,1.035924
...,...,...,...,...,...,...
8507,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22327989,5048.902326,1.045577,1.045772,1.046107
8508,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22335153,5048.902326,1.045815,1.045912,1.046169
8509,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22342307,5117.872032,1.045629,1.045839,1.046240
8510,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22349480,4975.260393,1.045822,1.045994,1.046303


In [17]:
limited_destination_states_df

,destination_vault_address,block,underlying_token_total_supply,underlying_safe_price,underlying_spot_price,underlying_backing
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,13157.451200,1.036523,1.036516,1.035855
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,13157.496413,1.036163,1.036121,1.035875
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,13157.496413,1.036462,1.036456,1.035906
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,13268.458946,1.036491,1.036477,1.035896
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,13268.526874,1.036501,1.036502,1.035924
...,...,...,...,...,...,...
8507,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22327989,5048.902326,1.045577,1.045772,1.046107
8508,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22335153,5048.902326,1.045815,1.045912,1.046169
8509,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22342307,5117.872032,1.045629,1.045839,1.046240
8510,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22349480,4975.260393,1.045822,1.045994,1.046303


In [18]:
raw_autopool_destination_state_df = pd.merge(
    limited_destination_states_df, autopool_destination_balance_of_df, on="block"
)

In [20]:
raw_autopool_destination_state_df.columns

Index([                                                                              'destination_vault_address',
                                                                                                         'block',
                                                                                 'underlying_token_total_supply',
                                                                                         'underlying_safe_price',
                                                                                         'underlying_spot_price',
                                                                                            'underlying_backing',
       ('0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', '0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', 'balanceOf'),
       ('0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', '0x865e59D439BF7310c9BC6117E6020B8C87De4065', 'balanceOf'),
       ('0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', '0x25cb41919d6B88e0D48108A4F5fe8FB

In [21]:
limited_destination_states_df

,destination_vault_address,block,underlying_token_total_supply,underlying_safe_price,underlying_spot_price,underlying_backing
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,13157.451200,1.036523,1.036516,1.035855
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,13157.496413,1.036163,1.036121,1.035875
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,13157.496413,1.036462,1.036456,1.035906
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,13268.458946,1.036491,1.036477,1.035896
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,13268.526874,1.036501,1.036502,1.035924
...,...,...,...,...,...,...
8507,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22327989,5048.902326,1.045577,1.045772,1.046107
8508,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22335153,5048.902326,1.045815,1.045912,1.046169
8509,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22342307,5117.872032,1.045629,1.045839,1.046240
8510,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22349480,4975.260393,1.045822,1.045994,1.046303


In [ ]:
# I really want

# autopool, destination_vault, balance_of, block

In [22]:
autopool_destination_balance_of_df

,"(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0xf3ae3c74EaD129e770A864CeE291A805b170bBe0, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0x865e59D439BF7310c9BC6117E6020B8C87De4065, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0xfB6f99FdF12E37Bfe3c4Cf81067faB10c465fb24, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0x8cA2201BC34780f14Bca452913ecAc8e9928d4cA, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0x896eCc16Ab4AFfF6cE0765A5B924BaECd7Fa455a, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0xC001f23397dB71B17602Ce7D90a983Edc38DB0d1, balanceOf)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, 0x6a8C6ff78082a2ae494EB9291DdC7254117298Ff, balanceOf)",...,"(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x2f2CC1bf461413014741dD68481dB4a3686DAC3D, balanceOf)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x40219bBda953ca811d2D0168Dc806a96b84791d9, balanceOf)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0xC9b5D82652a1C8214b0971A004983d0EEeDD751C, balanceOf)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0xf9779aEF9f77e78C857CB4A068c65CcBee25BAAc, balanceOf)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x60339056EC88996e41757E05a798310E46972cca, balanceOf)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x5c6aeb9ef0d5BbA4E6691f381003503FD0D45126, balanceOf)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xc4Eb861e7b66f593482a3D7E8adc314f6eEDA30B, balanceOf)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04, balanceOf)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xbA1462f43c6f60ebD1C62735c94E428aD073E01A, balanceOf)",block
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20759464
2024-09-16 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20766617
2024-09-17 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20773761
2024-09-18 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20780916
2024-09-19 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,389.430196,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20788084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-22 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22327989
2025-04-23 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22335153
2025-04-24 23:59:59+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22342307


In [ ]:
new_autopool_destination_states = []


def raw_autopool_destination_state_df(row: dict) -> None:

    for autopool_destination_vault_balance_of_value in row.keys():
        if autopool_destination_vault_balance_of_value not in [
            "destination_vault_address",
            "block",
            "underlying_token_total_supply",
            "underlying_safe_price",
            "underlying_spot_price",
            "underlying_backing",
        ]:

            autopool_vault_address, destination_vault_address, _ = autopool_destination_vault_balance_of_value
            autopool_balance_of = row[autopool_destination_vault_balance_of_value]

0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x865e59D439BF7310c9BC6117E6020B8C87De4065
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xfB6f99FdF12E37Bfe3c4Cf81067faB10c465fb24
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x8cA2201BC34780f14Bca452913ecAc8e9928d4cA
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x896eCc16Ab4AFfF6cE0765A5B924BaECd7Fa455a
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xC001f23397dB71B17602Ce7D90a983Edc38DB0d1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x6a8C6ff78082a2ae494EB9291DdC7254117298Ff
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xd96E943098B2AE81155e98D7DC8BeaB34C539f01
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xE382BBd32

In [6]:
autopool_to_all_ever_active_destinations.keys()

dict_keys(['0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', '0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5', '0xE800e3760FC20aA98c5df6A9816147f190455AF3', '0x35911af1B570E26f668905595dEd133D01CD3E5a'])

In [ ]:
autopool_to_all_ever_active_destinations.keys

In [3]:
# def build_pool_token_spot_price_calls(
#     chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
# ) -> list[Call]:

#     return [
#         Call(
#             ROOT_PRICE_ORACLE(chain),
#             ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
#             [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
#         )
#         for (pool_address, token_address) in zip(pool_addresses, token_addresses)
#     ]


# spot_price_calls = build_pool_token_spot_price_calls(
#     chain, full_destination_df["pool"], full_destination_df["token_address"]
# )
# destination_token_spot_price_df = get_raw_state_by_blocks(
#     spot_price_calls, possible_blocks, chain, include_block_number=True
# )
# destination_token_spot_price_df


# def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
#     return [
#         Call(
#             dest,
#             "underlyingReserves()(address[],uint256[])",
#             [
#                 (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
#                 (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
#             ],
#         )
#         for dest in destinations
#     ]


# underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])
# underlying_reserves_df = get_raw_state_by_blocks(
#     underlying_reserves_calls, possible_blocks, chain, include_block_number=True
# )
# underlying_reserves_df